In [1]:
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd
import time

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

In [3]:
def setup_driver():
  """Setup Chrome driver with options"""
  chrome_options = Options()
  chrome_options.page_load_strategy = "eager"

  service = Service(ChromeDriverManager().install())
  driver = webdriver.Chrome(service=service, options=chrome_options)
  driver.set_page_load_timeout(60)

  return driver

**States - Andhra Pradesh, Bihar, Gujarat**

In [4]:
states = ["ANDHRA PRADESH", "BIHAR", "GUJARAT"]

In [5]:
cols_names = {
    0: "Sno.",
    1: "Candidate",
    2: "Party",
    3: "Criminal Cases",
    4: "Education",
    5: "Age",
    6: "Total Assets",
    7: "Liabilities"
}
ua = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36"}

In [6]:
url_19 = "https://www.myneta.info/LokSabha2019/"
html_19 = requests.get(url_19, headers=ua, timeout=30)
print(html_19.status_code)

200


In [7]:
soup_19 = bs(html_19.text, "html.parser")

In [8]:
for card in soup_19.find_all("div", class_="w3-dropdown-click w3-block"):
    print(card.find("button").get_text(strip=True))
    break

ANDAMAN & NICOBAR ISLANDS


In [9]:
url_dict_19 = {}

In [10]:
found = 0
for card in soup_19.find_all("div", class_="w3-dropdown-click w3-block"):
    state = card.find("button").get_text(strip=True)
    if state not in states:
        continue
    found+=1
    url_dict_19[state] = card.find("a", class_="w3-bar-item w3-button w3-padding-small").get("href", "")
    if found == 3:
        break
print(url_dict_19)

{'ANDHRA PRADESH': 'index.php?action=show_constituencies&state_id=34', 'BIHAR': 'index.php?action=show_constituencies&state_id=37', 'GUJARAT': 'index.php?action=show_constituencies&state_id=39'}


In [11]:
data_19 = {}
for state in states:
    data_19[state] = []

In [12]:
driver = setup_driver()
try:
    for state in states:
        state_url = url_19+url_dict_19[state]
        while state_url:
            driver.get(state_url)
            WebDriverWait(driver, 10).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "table.w3-table"))
            )
            first_row = True
            for row in driver.find_elements(By.CSS_SELECTOR, "table.w3-table tr"):
                if first_row:
                    first_row = False
                    continue
                row_dic = {}
                i = 0
                for col in row.find_elements(By.TAG_NAME, "td"):
                    row_dic[i] = col.text.strip()
                    if (i == 6 or i == 7) and row_dic[i] == "":
                        row_dic[i] = col.find_element(By.TAG_NAME, "img").get_attribute("src")
                    i+=1
                data_19[state].append(row_dic)
            next_url = ""
            for page in driver.find_elements(By.CSS_SELECTOR, "center a"):
                if page.text.strip()=="Next":
                    next_url = page.get_attribute("href")
                    break
            if next_url:
                state_url = next_url
            else:
                state_url = ""
            print("Moving to the next page")
            time.sleep(2)
finally:
    driver.quit()

Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the ne

In [13]:
print(len(data_19["ANDHRA PRADESH"]))
print(len(data_19["BIHAR"]))
print(len(data_19["GUJARAT"]))

314
619
371


In [14]:
andhra_19 = pd.DataFrame(data_19["ANDHRA PRADESH"])
bihar_19 = pd.DataFrame(data_19["BIHAR"])
gujarat_19 = pd.DataFrame(data_19["GUJARAT"])

In [15]:
andhra_19 = andhra_19.rename(columns=cols_names)

In [16]:
bihar_19 = bihar_19.rename(columns=cols_names)

In [17]:
gujarat_19 = gujarat_19.rename(columns=cols_names)

In [18]:
andhra_19

,Sno.,Candidate,Party,Criminal Cases,Education,Age,Total Assets,Liabilities
0,1,A. Hemanth,IND,0,5th Pass,28,"Rs 50,000\n~ 50 Thou+",Rs 0\n~
1,2,Abdul Waris,SDPI,4,8th Pass,58,"Rs 51,10,000\n~ 51 Lacs+",Rs 0\n~
2,3,Adala Prabhakara Reddy,YSRCP,0,12th Pass,70,"Rs 2,21,16,63,364\n~ 221 Crore+","Rs 15,22,81,255\n~ 15 Crore+"
3,4,Adari Anand Kumar,TDP,0,10th Pass,43,"Rs 41,05,47,212\n~ 41 Crore+","Rs 20,55,91,400\n~ 20 Crore+"
4,5,Adinarayana Inty,BJP,1,Post Graduate,67,"Rs 3,38,85,898\n~ 3 Crore+","Rs 44,79,155\n~ 44 Lacs+"
...,...,...,...,...,...,...,...,...
309,310,Yanamadala Venkata Suresh,IND,0,Graduate,50,"Rs 13,02,000\n~ 13 Lacs+",Rs 0\n~
310,311,Yarlagadda Rama Mohana Rao,Bahujan Maha Party,0,12th Pass,56,"Rs 1,00,86,000\n~ 1 Crore+",Rs 0\n~
311,312,Yeduguri Sandinti Avinash Reddy,YSRCP,4,Post Graduate,34,"Rs 18,69,69,794\n~ 18 Crore+","Rs 2,28,33,319\n~ 2 Crore+"
312,313,Yella Rao Siyyadula,IND,0,12th Pass,37,"Rs 57,000\n~ 57 Thou+",Rs 0\n~


In [ ]:
bihar_19

In [ ]:
gujarat_19

In [ ]:
# andhra_19.to_csv("data/andhra_2019.csv", index=False)
# bihar_19.to_csv("data/bihar_2019.csv", index=False)
# gujarat_19.to_csv("data/gujarat_2019.csv", index=False)

In [ ]:
url_14 = "https://www.myneta.info/ls2014/"
html_14 = requests.get(url_14, headers=ua, timeout=30)
print(html_14.status_code)

In [ ]:
soup_14 = bs(html_14.text, "html.parser")

In [ ]:
url_dict_14 = {}
found = 0
for card in soup_14.find_all("div", class_="w3-dropdown-click w3-block"):
    state = card.find("button").get_text(strip=True)
    if state not in states:
        continue
    found+=1
    url_dict_14[state] = card.find("a", class_="w3-bar-item w3-button w3-padding-small").get("href", "")
    if found == 3:
        break
print(url_dict_14)

In [ ]:
data_14 = {}
for state in states:
    data_14[state] = []

In [ ]:
driver = setup_driver()
try:
    for state in states:
        state_url = url_14+url_dict_14[state]
        while state_url:
            driver.get(state_url)
            WebDriverWait(driver, 10).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "table.w3-table"))
            )
            first_row = True
            for row in driver.find_elements(By.CSS_SELECTOR, "table.w3-table tr"):
                if first_row:
                    first_row = False
                    continue
                row_dic = {}
                i = 0
                for col in row.find_elements(By.TAG_NAME, "td"):
                    row_dic[i] = col.text.strip()
                    if (i == 6 or i == 7) and row_dic[i] == "":
                        row_dic[i] = col.find_element(By.TAG_NAME, "img").get_attribute("src")
                    i+=1
                data_14[state].append(row_dic)
            next_url = ""
            for page in driver.find_elements(By.CSS_SELECTOR, "center a"):
                if page.text.strip()=="Next":
                    next_url = page.get_attribute("href")
                    break
            if next_url:
                state_url = next_url
            else:
                state_url = ""
            print("Moving to the next page")
            time.sleep(2)
finally:
    driver.quit()

In [ ]:
print(len(data_14["ANDHRA PRADESH"]))
print(len(data_14["BIHAR"]))
print(len(data_14["GUJARAT"]))

In [ ]:
andhra_14 = pd.DataFrame(data_14["ANDHRA PRADESH"])
bihar_14 = pd.DataFrame(data_14["BIHAR"])
gujarat_14 = pd.DataFrame(data_14["GUJARAT"])

In [ ]:
andhra_14 = andhra_14.rename(columns=cols_names)
bihar_14 = bihar_14.rename(columns=cols_names)
gujarat_14 = gujarat_14.rename(columns=cols_names)

In [ ]:
andhra_14

In [ ]:
bihar_14

In [ ]:
gujarat_14

In [ ]:
# andhra_14.to_csv("data/andhra_2014.csv", index=False)
# bihar_14.to_csv("data/bihar_2014.csv", index=False)
# gujarat_14.to_csv("data/gujarat_2014.csv", index=False)

Scraping the winner list from wikipedia 